# 04 Fusion Pack Grouping — family и фасовки

Эта тетрадка — следующий шаг после `03_matching_comparison.ipynb`.
Здесь мы не запускаем модели заново. Мы берём готовые `score` и threshold-решения из compact CSV, выбираем один fusion-run по `dev`, а потом собираем два уровня групп.


## Что здесь происходит

- `family` — один базовый товар. `200 г` и `3 x 200 г` могут оказаться в одной family, если matching-модель решила, что это тот же продукт.
- `pack` — конкретная фасовка внутри family. Она отделяется deterministic правилами по `unit_amount`, `total_amount`, `multipack_count`.
- Выбор метода и threshold strategy делается только по `dev`. `test` можно смотреть как честную проверку, но не как источник выбора.


## Блок кода 1. Подготовка окружения

Импортируем только research helpers и задаём корень проекта. Если notebook открыт из папки `notebooks/`, код всё равно найдёт пакет `research.dedup`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:  # графики необязательны для CSV-артефактов
    plt = None

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "research" / "dedup").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    ComponentConfig,
    add_component_flags,
    build_components,
    component_size_summary,
    prepare_fusion_pair_edges,
    select_fusion_run,
)

PROJECT_ROOT


## Блок кода 2. Пути и настройки

**Если не уверен, оставь параметры как есть.** Дефолтный режим выбирает fusion-run по `dev` с учётом веса продаж, а качество показывает на `test`.

Что приходит на вход:

- `SUMMARY_PATH` — таблица качества из `03`: какой `method`, какой `threshold_strategy`, какой порог и какая цена ошибок.
- `PREDICTIONS_PATH` — построчные предсказания пар из `03`: score, true label, predicted label и pack/weight-поля.

Что notebook сохранит:

- `COMPONENTS_PATH` — товары с номерами групп `fusion_family_id` и `fusion_pack_id`.
- `PAIR_EVAL_PATH` — пары с проверкой, попали ли они в одну family/pack-группу.

Главные настройки лежат прямо в следующей ячейке. Окружение/env здесь не используется как скрытый override: что написано в `MY_*`, то и работает.

| Параметр | Значение по умолчанию | Какие значения ставить | Что это меняет |
| --- | --- | --- | --- |
| `MY_FUSION_METHOD` | `None` | `None`, `"rule_based_fuzzy"`, `"bi_encoder_zero_shot"`, `"cross_encoder_zero_shot"`, `"reranker_bge_v2_m3"`, `"reranker_qwen3_4b"`, `"reranker_qwen3_0_6b"`, `"reranker_jina_v3"` | Какой scorer/reranker использовать для склейки family. `None` = выбрать method автоматически по `dev`. |
| `MY_FUSION_THRESHOLD_STRATEGY` | `None` | `None` | Рекомендуемый режим. Сначала берёт `threshold_weighted_cost`, то есть минимальную цену ошибок с весом продаж. Если weighted-строк нет, откатывается на `threshold_cost_sensitive`. |
| `MY_FUSION_THRESHOLD_STRATEGY` | вручную | `"threshold_weighted_cost"` | Жёстко взять sales-weighted cost. Ошибки на SKU с большим `Продажи, шт` весят сильнее. Если weighted-стратегии нет в файле, notebook упадёт, и это нормально: значит `03` не подтянул веса. |
| `MY_FUSION_THRESHOLD_STRATEGY` | вручную | `"threshold_cost_sensitive"` | Обычная цена ошибок без продаж: false merge дороже false split, но каждая пара весит одинаково. Нужен как fallback или sanity-check. |
| `MY_FUSION_THRESHOLD_STRATEGY` | вручную | `"threshold_max_weighted_f1"` | Максимальный weighted F1. Полезно для диагностики, но для группировки хуже дефолта: может быть смелее по склейкам. |
| `MY_FUSION_THRESHOLD_STRATEGY` | вручную | `"threshold_max_f1"` | Максимальный обычный F1 без веса продаж. Это исследовательский режим, не основной downstream-дефолт. |
| `MY_FUSION_EVAL_SPLIT` | `"test"` | `"test"`, `"dev"`, `"all"` | Какую часть показывать в диагностике. `test` — честная проверка; `dev` — где подбирался threshold; `all` — всё вместе для просмотра. |


In [ ]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

SUMMARY_PATH = REPORTS_DIR / "binary_threshold_summary.csv"
PREDICTIONS_PATH = REPORTS_DIR / "binary_threshold_predictions.csv"
COMPONENTS_PATH = DATA_DIR / "fusion_components_sauces.csv"
PAIR_EVAL_PATH = DATA_DIR / "fusion_pair_eval_sauces.csv"

# НАСТРОЙКИ ПОЛЬЗОВАТЕЛЯ
# None = выбрать method автоматически по dev. Чтобы зафиксировать method, впиши строку из списка ниже.
MY_FUSION_METHOD = None

# None = рекомендуемый режим:
# 1) threshold_weighted_cost, если 03 подтянул веса продаж;
# 2) threshold_cost_sensitive, если weighted-строк нет.
MY_FUSION_THRESHOLD_STRATEGY = None

# "test" = честная диагностика после выбора на dev. Можно поставить "dev" или "all".
MY_FUSION_EVAL_SPLIT = "test"

EVAL_SPLIT = MY_FUSION_EVAL_SPLIT
if EVAL_SPLIT not in {"test", "dev", "all"}:
    raise ValueError("MY_FUSION_EVAL_SPLIT должен быть 'test', 'dev' или 'all'")

print("Файлы:")
print(f"- summary из 03: {SUMMARY_PATH}")
print(f"- predictions из 03: {PREDICTIONS_PATH}")
print(f"- результат по товарам: {COMPONENTS_PATH}")
print(f"- результат по парам: {PAIR_EVAL_PATH}")

print("\nТекущие настройки из notebook:")
print(f"- MY_FUSION_METHOD: {MY_FUSION_METHOD or 'None -> auto: лучший method по dev'}")
print(
    "- MY_FUSION_THRESHOLD_STRATEGY: "
    f"{MY_FUSION_THRESHOLD_STRATEGY or 'None -> threshold_weighted_cost, если есть; иначе threshold_cost_sensitive'}"
)
print(f"- MY_FUSION_EVAL_SPLIT: {EVAL_SPLIT!r}  # варианты: 'test', 'dev', 'all'")

if SUMMARY_PATH.exists():
    settings_preview = pd.read_csv(SUMMARY_PATH)
    available_methods = sorted(settings_preview["method"].dropna().astype(str).unique())
    available_strategies = sorted(settings_preview["threshold_strategy"].dropna().astype(str).unique())
    print("\nДоступные значения MY_FUSION_METHOD в текущем файле:")
    for method in available_methods:
        print(f"- {method}")
    print("\nДоступные значения MY_FUSION_THRESHOLD_STRATEGY в текущем файле:")
    for strategy in available_strategies:
        print(f"- {strategy}")
else:
    print("\nПока нет binary_threshold_summary.csv. Сначала запусти notebook 03.")


## Блок кода 3. Загрузка и выбор fusion-run

Здесь notebook применяет настройки из блока выше.

Логика выбора:

1. Если `MY_FUSION_METHOD` задан, берём только этот method. Если `None`, method выбирается автоматически по строкам `dev`.
2. Если `MY_FUSION_THRESHOLD_STRATEGY` задан, берём только эту strategy.
3. Если `MY_FUSION_THRESHOLD_STRATEGY = None`, сначала пробуем `threshold_weighted_cost`: это режим, где сам threshold подобран с учётом веса продаж.
4. Если `threshold_weighted_cost` отсутствует, значит `03` не смог надёжно подтянуть веса продаж; тогда используем fallback `threshold_cost_sensitive`.
5. Среди methods выбранной strategy ранжируем по `cost`, затем `false_merge_count`, затем `false_split_count`, затем `f1`. Это не даёт rule-based победить qwen только из-за микроскопической разницы weighted-cost на dev.

`test` здесь не участвует в выборе. Он нужен ниже, чтобы честно посмотреть качество выбранного варианта.


In [ ]:
if not SUMMARY_PATH.exists() or not PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        "Run notebooks/03_matching_comparison.ipynb first: expected "
        f"{SUMMARY_PATH.name} and {PREDICTIONS_PATH.name} in {REPORTS_DIR}."
    )

threshold_summary = pd.read_csv(SUMMARY_PATH)
threshold_predictions = pd.read_csv(PREDICTIONS_PATH)

fusion_run = select_fusion_run(
    threshold_summary,
    method=MY_FUSION_METHOD,
    threshold_strategy=MY_FUSION_THRESHOLD_STRATEGY,
    split="dev",
)
selected_summary = threshold_summary[
    threshold_summary["method"].astype(str).eq(fusion_run.method)
    & threshold_summary["threshold_strategy"].astype(str).eq(fusion_run.threshold_strategy)
].copy()

print("Выбранный fusion-run:")
print(f"- method: {fusion_run.method}")
print(f"- threshold_strategy: {fusion_run.threshold_strategy}")
print(f"- threshold_same: {fusion_run.threshold_same:.6f}")

summary_columns = [
    "method",
    "split",
    "threshold_strategy",
    "threshold_same",
    "precision",
    "recall",
    "f1",
    "weighted_f1",
    "false_merge_count",
    "false_split_count",
    "cost",
    "weighted_false_merge_cost",
    "weighted_total_cost",
    "weight_source",
]
summary_columns = [column for column in summary_columns if column in selected_summary.columns]

display(selected_summary[summary_columns].sort_values("split"))
fusion_run


## Блок кода 4. Family/pack edge flags

Здесь каждая пара получает два ответа:

- `fusion_family_edge`: модель сказала, что это один базовый товар;
- `fusion_pack_edge`: это один базовый товар и deterministic pack signature совпал.


In [ ]:
fusion_pairs = prepare_fusion_pair_edges(threshold_predictions, fusion_run)

edge_summary = pd.DataFrame(
    [
        {"edge": "family", "positive_pairs": int(fusion_pairs["fusion_family_edge"].sum())},
        {"edge": "pack", "positive_pairs": int(fusion_pairs["fusion_pack_edge"].sum())},
        {"edge": "same_pack_signature", "positive_pairs": int(fusion_pairs["same_pack_signature"].sum())},
    ]
)
display(edge_summary)

if EVAL_SPLIT == "all":
    eval_pairs = fusion_pairs.copy()
else:
    eval_pairs = fusion_pairs[fusion_pairs["split"].astype(str).eq(EVAL_SPLIT)].copy()
    if eval_pairs.empty:
        raise ValueError(f"Нет fusion pairs для MY_FUSION_EVAL_SPLIT={EVAL_SPLIT!r}")
print(f"Fusion pairs: {len(fusion_pairs)} total, {len(eval_pairs)} in diagnostic split={EVAL_SPLIT!r}")


## Блок кода 5. Сборка component-групп

Connected components склеивают товары транзитивно: если A связан с B, а B связан с C, они попадают в одну group. Для pack-группы используется более строгий edge.


In [ ]:
FAMILY_EDGE_LABELS = {"exact_duplicate"}
PACK_EDGE_LABELS = {"exact_duplicate"}

pred_family_config = ComponentConfig(label_col="predicted_family_label", component_col="fusion_family_id")
pred_pack_config = ComponentConfig(label_col="predicted_pack_label", component_col="fusion_pack_id")
true_family_config = ComponentConfig(label_col="true_family_label", component_col="true_family_id")
true_pack_config = ComponentConfig(label_col="true_pack_label", component_col="true_pack_id")

pred_family_components = build_components(fusion_pairs, edge_labels=FAMILY_EDGE_LABELS, config=pred_family_config)
pred_pack_components = build_components(fusion_pairs, edge_labels=PACK_EDGE_LABELS, config=pred_pack_config)
true_family_components = build_components(fusion_pairs, edge_labels=FAMILY_EDGE_LABELS, config=true_family_config)
true_pack_components = build_components(fusion_pairs, edge_labels=PACK_EDGE_LABELS, config=true_pack_config)

def _component_overview(components: pd.DataFrame, component_col: str, graph: str) -> dict[str, object]:
    sizes = component_size_summary(components, component_col=component_col)
    return {
        "graph": graph,
        "components": int(sizes[component_col].nunique()) if not sizes.empty else 0,
        "multi_node_components": int((sizes["nodes"] > 1).sum()) if not sizes.empty else 0,
        "max_nodes": int(sizes["nodes"].max()) if not sizes.empty else 0,
    }

component_overview = pd.DataFrame(
    [
        _component_overview(pred_family_components, "fusion_family_id", "pred_family"),
        _component_overview(pred_pack_components, "fusion_pack_id", "pred_pack"),
        _component_overview(true_family_components, "true_family_id", "true_family_partial"),
        _component_overview(true_pack_components, "true_pack_id", "true_pack_partial"),
    ]
)
display(component_overview)
display(component_size_summary(pred_family_components, component_col="fusion_family_id").head(15))
display(component_size_summary(pred_pack_components, component_col="fusion_pack_id").head(15))


## Блок кода 6. Быстрые визуализации групп

График показывает, не появился ли один слишком большой компонент. Для SKU dedup это тревожный сигнал: один false merge может сцепить много разных товаров.


In [ ]:
if plt is None:
    print("matplotlib is not available; skipping charts")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, components, col, title in [
        (axes[0], pred_family_components, "fusion_family_id", "Predicted family sizes"),
        (axes[1], pred_pack_components, "fusion_pack_id", "Predicted pack sizes"),
    ]:
        sizes = component_size_summary(components, component_col=col).head(15)
        if sizes.empty:
            ax.set_title(title)
            ax.text(0.5, 0.5, "no components", ha="center", va="center")
            continue
        ax.bar(range(len(sizes)), sizes["nodes"])
        ax.set_title(title)
        ax.set_xlabel("top components")
        ax.set_ylabel("nodes")
    plt.tight_layout()
    plt.show()


## Блок кода 7. Примеры: одна family, но разная фасовка

Это главный смысл шага: модель склеивает базовый товар, а pack-правила не дают смешать `200 г`, `3 x 200 г`, `500 г` в одну фасовку.


In [ ]:
example_cols = [
    "split",
    "score",
    "same_base_product",
    "predicted_binary",
    "same_pack_signature",
    "title_a",
    "title_b",
    "brand_a",
    "brand_b",
    "unit_amount_a",
    "unit_amount_b",
    "total_amount_a",
    "total_amount_b",
    "multipack_count_a",
    "multipack_count_b",
]
available_cols = [column for column in example_cols if column in fusion_pairs.columns]
same_family_different_pack = fusion_pairs[
    fusion_pairs["fusion_family_edge"] & ~fusion_pairs["fusion_pack_edge"]
].copy()

display(same_family_different_pack[available_cols].head(20))


## Блок кода 8. Сохранение compact artifacts

Эти два CSV становятся входом для следующих notebooks. Они не заменяют production-данные и не пишут ничего в БД.


In [ ]:
def _node_catalog(pairs: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for side in ["a", "b"]:
        mapping = {
            "node_id": f"raw_record_id_{side}",
            "marketplace": f"marketplace_{side}",
            "sku": f"sku_{side}",
            "brand": f"brand_{side}",
            "title": f"title_{side}",
            "unit_amount": f"unit_amount_{side}",
            "total_amount": f"total_amount_{side}",
            "multipack_count": f"multipack_count_{side}",
            "sales_volume": f"sales_volume_{side}",
        }
        present = {target: source for target, source in mapping.items() if source in pairs.columns}
        if "node_id" not in present:
            continue
        part = pairs[list(present.values())].rename(columns={source: target for target, source in present.items()})
        rows.extend(part.to_dict("records"))
    if not rows:
        return pd.DataFrame(columns=["node_id"])
    return pd.DataFrame(rows).dropna(subset=["node_id"]).drop_duplicates("node_id").reset_index(drop=True)

pair_eval = add_component_flags(
    fusion_pairs,
    true_family_components,
    config=true_family_config,
    same_component_col="true_same_family",
)
pair_eval = add_component_flags(
    pair_eval,
    pred_family_components,
    config=pred_family_config,
    same_component_col="pred_same_family",
)
pair_eval = add_component_flags(
    pair_eval,
    true_pack_components,
    config=true_pack_config,
    same_component_col="true_same_pack",
)
pair_eval = add_component_flags(
    pair_eval,
    pred_pack_components,
    config=pred_pack_config,
    same_component_col="pred_same_pack",
)

components_export = (
    _node_catalog(fusion_pairs)
    .merge(pred_family_components, on="node_id", how="left")
    .merge(pred_pack_components, on="node_id", how="left")
    .merge(true_family_components, on="node_id", how="left")
    .merge(true_pack_components, on="node_id", how="left")
)
components_export["fusion_method"] = fusion_run.method
components_export["fusion_threshold_strategy"] = fusion_run.threshold_strategy
components_export["fusion_threshold_same"] = fusion_run.threshold_same

DATA_DIR.mkdir(parents=True, exist_ok=True)
components_export.to_csv(COMPONENTS_PATH, index=False)
pair_eval.to_csv(PAIR_EVAL_PATH, index=False)

display(components_export.head(20))
print(f"Saved components: {COMPONENTS_PATH} ({len(components_export)} rows)")
print(f"Saved pair eval: {PAIR_EVAL_PATH} ({len(pair_eval)} rows)")


## Что делать после этой тетрадки

Запустите `05_evaluation_report.ipynb`, чтобы посмотреть качество matching/fusion и graph diagnostics. Затем `06_grouped_sku_demo.ipynb` покажет склеенные SKU-группы на реальных строках DuckDB.
